In [25]:
import os
import glob
import pickle
import numpy as np

SOURCE = 'labeled_data/MECO/video'

# ---------- 1. Find pickle files ----------
if os.path.isfile(SOURCE):
    files = [SOURCE]
else:
    files = sorted(
        glob.glob(os.path.join(SOURCE, '**', '*.pkl'), recursive=True)
        + glob.glob(os.path.join(SOURCE, '**', '*.pickle'), recursive=True)
        + glob.glob(os.path.join(SOURCE, '**', '*.pckl'), recursive=True)
    )

print(f"Found {len(files)} pickle file(s):")
for f in files[:42]:
    print("  ", f, f"({os.path.getsize(f)/1e6:.2f} MB)")
if not files:
    raise SystemExit("No pickle files found, check SOURCE path.")


# ---------- 2. Loader (falls back to pandas / latin1 for old py2 pickles) ----------
def load_pickle(path):
    try:
        with open(path, 'rb') as fh:
            return pickle.load(fh)
    except Exception as e1:
        try:
            import pandas as pd
            return pd.read_pickle(path)
        except Exception as e2:
            with open(path, 'rb') as fh:
                return pickle.load(fh, encoding='latin1')


# ---------- 3. Recursive inspector ----------
def describe(obj, name='root', depth=0, max_depth=4, max_items=15):
    pad = '  ' * depth
    t = type(obj).__name__

    try:
        import pandas as pd
    except ImportError:
        pd = None

    if pd is not None and isinstance(obj, pd.DataFrame):
        print(f"{pad}{name}: DataFrame shape={obj.shape}")
        print(f"{pad}  index: {obj.index.dtype}, {obj.index[:3].tolist()} ...")
        print(f"{pad}  columns ({len(obj.columns)}): {list(obj.columns)}")
        print(f"{pad}  dtypes:\n{obj.dtypes.value_counts().to_string()}")
        print(f"{pad}  head:\n{obj.head().to_string()}")
        au_cols = [c for c in obj.columns if 'au' in str(c).lower()]
        if au_cols:
            print(f"{pad}  AU-like columns ({len(au_cols)}): {au_cols}")
        print(f"{pad}  NaNs per column (non-zero only):")
        nans = obj.isna().sum()
        print(nans[nans > 0].to_string() if (nans > 0).any() else f"{pad}    none")
        print(f"{pad}  describe:\n{obj.describe().T.head(30).to_string()}")

    elif pd is not None and isinstance(obj, pd.Series):
        print(f"{pad}{name}: Series len={len(obj)} dtype={obj.dtype}")
        print(obj.head().to_string())

    elif isinstance(obj, np.ndarray):
        print(f"{pad}{name}: ndarray shape={obj.shape} dtype={obj.dtype}")
        if obj.size and np.issubdtype(obj.dtype, np.number):
            print(f"{pad}  min={np.nanmin(obj):.4g} max={np.nanmax(obj):.4g} "
                  f"mean={np.nanmean(obj):.4g} nans={np.isnan(obj).sum()}")

    elif isinstance(obj, dict):
        print(f"{pad}{name}: dict with {len(obj)} keys")
        keys = list(obj.keys())
        print(f"{pad}  keys (first {max_items}): {keys[:max_items]}")
        if depth < max_depth:
            for k in keys[:max_items]:
                describe(obj[k], name=f"[{k!r}]", depth=depth + 1,
                         max_depth=max_depth, max_items=max_items)

    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{name}: {t} len={len(obj)}")
        if depth < max_depth:
            for i, v in enumerate(obj[:3]):
                describe(v, name=f"[{i}]", depth=depth + 1,
                         max_depth=max_depth, max_items=max_items)

    else:
        r = repr(obj)
        print(f"{pad}{name}: {t} -> {r[:100]}{'...' if len(r) > 100 else ''}")


# ---------- 4. Run on each file ----------
# for path in files[0:1]:   # change/remove the slice to inspect more
path = files[0]
print("\n" + "=" * 80)
print("FILE:", path)
print("=" * 80)
data = load_pickle(path)
describe(data)

Found 42 pickle file(s):
   labeled_data/MECO/video/video_S1.pkl (28.18 MB)
   labeled_data/MECO/video/video_S10.pkl (27.56 MB)
   labeled_data/MECO/video/video_S11.pkl (27.56 MB)
   labeled_data/MECO/video/video_S12.pkl (27.37 MB)
   labeled_data/MECO/video/video_S13.pkl (24.07 MB)
   labeled_data/MECO/video/video_S14.pkl (23.23 MB)
   labeled_data/MECO/video/video_S15.pkl (27.41 MB)
   labeled_data/MECO/video/video_S16.pkl (27.56 MB)
   labeled_data/MECO/video/video_S17.pkl (24.29 MB)
   labeled_data/MECO/video/video_S18.pkl (27.67 MB)
   labeled_data/MECO/video/video_S19.pkl (27.63 MB)
   labeled_data/MECO/video/video_S2.pkl (27.63 MB)
   labeled_data/MECO/video/video_S20.pkl (21.32 MB)
   labeled_data/MECO/video/video_S21.pkl (27.63 MB)
   labeled_data/MECO/video/video_S22.pkl (27.74 MB)
   labeled_data/MECO/video/video_S23.pkl (27.92 MB)
   labeled_data/MECO/video/video_S24.pkl (27.63 MB)
   labeled_data/MECO/video/video_S25.pkl (27.56 MB)
   labeled_data/MECO/video/video_S26.pkl 

In [26]:
print(data['emotion_to_idx_sa']) # Can chose which session to inspect, e.g. session1, session2, session3 or emotion_to_idx_sa
print('--------------------------------')

s1 = data['session1']
s1_keys = list(s1.keys())
print(s1_keys)
print(s1)

{'angry': 0, 'boredom': 1, 'sadness': 2, 'neutral': 3, 'tension': 4, 'happiness': 5}
--------------------------------
['deep_feature', 'au_openface', 'hp_openface', 'gz_openface', 'label_vi_class', 'label_valence', 'label_arousal', 'emotion_to_idx_vi', 'label_sa_class']
{'deep_feature': [tensor([[[ 0.8795, -1.7685,  0.8400,  ..., -1.0537, -2.1933,  0.5585],
         [ 0.8793, -1.7690,  0.8399,  ..., -1.0535, -2.1938,  0.5584],
         [ 0.8799, -1.7631,  0.8455,  ..., -1.0576, -2.1933,  0.5569],
         ...,
         [ 0.8938, -1.7741,  0.8422,  ..., -1.0549, -2.2036,  0.5504],
         [ 0.8898, -1.7711,  0.8414,  ..., -1.0528, -2.1993,  0.5537],
         [ 0.8900, -1.7696,  0.8416,  ..., -1.0535, -2.1985,  0.5537]]]), tensor([[[ 0.8898, -1.7711,  0.8414,  ..., -1.0528, -2.1993,  0.5537],
         [ 0.8900, -1.7696,  0.8416,  ..., -1.0535, -2.1985,  0.5537],
         [ 0.8783, -1.7751,  0.8466,  ..., -1.0514, -2.2040,  0.5553],
         ...,
         [ 0.8797, -1.7731,  0.8439,  ...

In [27]:
au = s1['au_openface']
# au_keys = list(s1.keys())
# print(s1_keys)
print('Length of action units:', len(au))
print(au)


Length of action units: 247
[tensor([[0.0000, 0.0000, 1.3600, 0.0000, 2.5800, 2.6500, 1.1000, 2.4000, 0.8600,
         0.9100, 1.3900, 2.6500, 0.6800, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 1.0000, 0.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 0.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.9900, 0.0000, 2.1600, 2.1900, 0.8200, 1.9300, 0.6800,
         0.7300, 0.9500, 2.2500, 0.3600, 0.0800, 0.0800, 0.0000, 0.0800, 0.0000,
         0.0000, 1.0000, 0.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 1.0000,
         1.0000, 1.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5400, 0.0500, 1.7500, 1.8500, 0.4600, 1.5000, 0.5300,
         0.4300, 0.6700, 1.8900, 0.1400, 0.0800, 0.0800, 0.1200, 0.1500, 0.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 1.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0

In [33]:
print(au[0].shape)

torch.Size([16, 35])


In [29]:
labels_vi = s1['label_vi_class']
labels_sa = s1['label_sa_class']
emotions_to_idx = s1['emotion_to_idx_vi']

print(labels_vi)
print(labels_sa)
print(emotions_to_idx)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [34]:
"""
Loading + preprocessing for MECO window-level features.

Assumed layout per subject dict `d` (adapt KEYS to what `describe(d)` prints):
    d[<au key>] : (N, 16, 35)   16 frames x 35 OpenFace AUs per 4 s window
    d[<hp key>] : (N, 16, 6)    head pose (optional)
    d[<eg key>] : (N, 16, 2)    gaze angles (optional)
    d['label_vi_class'], d['label_sa_class'] : (N,)
"""
import numpy as np
import torch
from torch.utils.data import Dataset


def describe(d):
    """Print every key with its type/shape so you can fill in KEYS."""
    for k, v in d.items():
        shape = getattr(v, "shape", None)
        if shape is None and isinstance(v, (list, tuple)):
            shape = ("list", len(v))
        print(f"{k:30s} {type(v).__name__:12s} {shape}")


def trial_ids_from_labels(y_vi):
    """Consecutive windows with the same stimulus label = one trial.
    (Each session has one trial per emotion, so neighbouring trials differ.)"""
    y = np.asarray(y_vi)
    return np.concatenate([[0], np.cumsum(y[1:] != y[:-1])])


def load_subject(d, subject_id, keys, label_key="label_vi_class", label_map=None):
    """keys: {'au': '<key in d>', 'hp': '<key>', 'eg': '<key>'} (hp/eg optional)."""
    out = {m: np.asarray(d[k], dtype=np.float32) for m, k in keys.items()}
    y = np.asarray(d[label_key]).astype(int)
    if label_map is not None:  # e.g. 6-class self-report -> 5-class (boredom into neutral)
        y = np.array([label_map[v] for v in y])
    for m, a in out.items():
        assert a.ndim == 3 and a.shape[0] == len(y), f"{m}: expected (N,T,F), got {a.shape}"
    out["y"] = y
    out["subject"] = np.full(len(y), subject_id)
    out["trial"] = trial_ids_from_labels(d["label_vi_class"])
    return out


def concat_subjects(parts):
    """Stack per-subject dicts; make trial ids globally unique."""
    offset, fixed = 0, []
    for p in parts:
        p = dict(p)
        p["trial"] = p["trial"] + offset
        offset = p["trial"].max() + 1
        fixed.append(p)
    return {k: np.concatenate([p[k] for p in fixed]) for k in fixed[0]}


def normalize_per_subject(data, mods=("au", "hp", "eg")):
    """z-score each feature per subject. Binary columns (OpenFace AU_c presence
    flags) are left untouched. Uses only the subject's own statistics."""
    for s in np.unique(data["subject"]):
        idx = np.where(data["subject"] == s)[0]
        for m in mods:
            if m not in data:
                continue
            a = data[m][idx]
            flat = a.reshape(-1, a.shape[-1])
            mu, sd = np.nanmean(flat, 0), np.nanstd(flat, 0) + 1e-6
            binary = np.array([np.isin(np.unique(c[~np.isnan(c)]), [0, 1]).all() for c in flat.T])
            mu[binary], sd[binary] = 0.0, 1.0
            data[m][idx] = (a - mu) / sd
    return data


class MecoDataset(Dataset):
    """Returns ({mod: (T,F)}, missing_mask (T,), label).
    NaN frames (failed face detection) are zero-filled and flagged in the mask."""

    def __init__(self, data, idx, mods=("au", "hp", "eg"), train=False,
                 noise=0.05, frame_drop=0.1):
        self.data, self.idx, self.mods = data, np.asarray(idx), [m for m in mods if m in data]
        self.train, self.noise, self.frame_drop = train, noise, frame_drop

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = {m: torch.from_numpy(self.data[m][j]) for m in self.mods}
        T = next(iter(x.values())).shape[0]
        miss = torch.zeros(T, dtype=torch.bool)
        for m in self.mods:
            miss |= torch.isnan(x[m]).any(-1)
            x[m] = torch.nan_to_num(x[m])
        if self.train:
            for m in self.mods:
                x[m] = x[m] + self.noise * torch.randn_like(x[m])
            miss |= torch.rand(T) < self.frame_drop
            if miss.all():
                miss[torch.randint(T, (1,))] = False
        return x, miss, int(self.data["y"][j])


def make_synthetic(n_subjects=8, n_trials=6, wins_per_trial=20, T=16, n_classes=6, seed=0):
    """Random data with the same layout, for smoke-testing the pipeline."""
    rng = np.random.default_rng(seed)
    parts = []
    for s in range(n_subjects):
        y = np.repeat(rng.permutation(n_classes)[:n_trials], wins_per_trial)
        au = rng.normal(size=(len(y), T, 35)).astype(np.float32)
        au[..., 17:] = (rng.random((len(y), T, 18)) > 0.7).astype(np.float32)  # binary AU_c
        au[..., 0] += y[:, None] * 0.5                                         # weak signal
        d = {"au": au, "hp": rng.normal(size=(len(y), T, 6)).astype(np.float32),
             "eg": rng.normal(size=(len(y), T, 2)).astype(np.float32),
             "label_vi_class": y, "label_sa_class": y}
        parts.append(load_subject(d, s, {"au": "au", "hp": "hp", "eg": "eg"}))
    return concat_subjects(parts)

In [39]:
"""
S2D-inspired model for MECO window features.

  frames (B,16,F_m) per modality
    -> per-frame "static" encoder per modality (MLP or 1-D conv over the AU axis)
    -> concat modality embeddings + linear fusion -> (B,16,d)
    -> [CLS] + learned positions -> temporal Transformer
    -> CLS -> classifier
"""
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader

# from meco_data import MecoDataset


class FrameEncoder(nn.Module):
    """Encodes one modality's per-frame feature vector (F) into d dims."""

    def __init__(self, in_dim, d, kind="mlp", drop=0.1):
        super().__init__()
        self.kind = kind
        if kind == "mlp":
            self.net = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, d), nn.GELU(),
                                     nn.Dropout(drop), nn.Linear(d, d))
        elif kind == "conv":  # conv over the AU index axis; weak inductive bias, see notes
            self.conv = nn.Sequential(nn.Conv1d(1, 16, 3, padding=1), nn.GELU(),
                                      nn.Conv1d(16, 32, 3, padding=1), nn.GELU(),
                                      nn.AdaptiveAvgPool1d(8), nn.Flatten())
            self.proj = nn.Linear(32 * 8, d)
        else:
            raise ValueError(kind)
        self.norm = nn.LayerNorm(d)

    def forward(self, x):  # (B,T,F) -> (B,T,d)
        if self.kind == "mlp":
            return self.norm(self.net(x))
        B, T, F = x.shape
        h = self.proj(self.conv(x.reshape(B * T, 1, F)))
        return self.norm(h).reshape(B, T, -1)


class S2DLite(nn.Module):
    def __init__(self, dims, n_classes, d=128, depth=2, heads=4, T=16,
                 drop=0.2, au_encoder="mlp", modality_drop=0.0):
        super().__init__()
        self.mods = list(dims)
        self.enc = nn.ModuleDict({m: FrameEncoder(f, d, au_encoder if m == "au" else "mlp")
                                  for m, f in dims.items()})
        self.fuse = nn.Sequential(nn.Linear(d * len(dims), d), nn.LayerNorm(d))
        self.modality_drop = modality_drop
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, T + 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.temporal = nn.TransformerEncoder(layer, depth)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(d, n_classes))

    def forward(self, x, missing=None):
        """x: {mod: (B,T,F)}, missing: (B,T) bool, True = frame unusable."""
        zs = []
        for m in self.mods:
            z = self.enc[m](x[m])
            if self.training and self.modality_drop > 0 and m != "au":  # optional regulariser
                keep = (torch.rand(z.shape[0], 1, 1, device=z.device) > self.modality_drop).float()
                z = z * keep
            zs.append(z)
        z = self.fuse(torch.cat(zs, -1))
        B, T, _ = z.shape
        z = torch.cat([self.cls.expand(B, -1, -1), z], 1) + self.pos[:, : T + 1]
        pad = None
        if missing is not None:
            pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=z.device), missing], 1)
        z = self.temporal(z, src_key_padding_mask=pad)
        return self.head(self.norm(z[:, 0]))


# ----------------------------------------------------------------- training
def _to(x, miss, y, dev):
    return {k: v.to(dev) for k, v in x.items()}, miss.to(dev), y.to(dev)


@torch.no_grad()
def evaluate(model, loader, dev):
    model.eval()
    ys, ps = [], []
    for x, miss, y in loader:
        x, miss, y = _to(x, miss, y, dev)
        ps.append(model(x, miss).argmax(-1).cpu())
        ys.append(y.cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return {"UAR": recall_score(y, p, average="macro", zero_division=0),
            "WAR": accuracy_score(y, p)}


def train_cv(data, n_classes, mods=("au", "hp", "eg"), folds=5, epochs=30, bs=64,
             lr=3e-4, wd=1e-2, seed=0, dev=None, **model_kw):
    """Subject-wise K-fold (SI protocol). ~15% of the training subjects are held
    out for model selection so the test fold is never used to pick an epoch."""
    dev = dev or ("cuda" if torch.cuda.is_available() else "cpu")
    mods = [m for m in mods if m in data]
    dims = {m: data[m].shape[-1] for m in mods}
    T = data[mods[0]].shape[1]
    subj, rng = data["subject"], np.random.default_rng(seed)
    results = []
    for f, (tr, te) in enumerate(GroupKFold(folds).split(np.zeros(len(subj)), groups=subj)):
        tr_subj = rng.permutation(np.unique(subj[tr]))
        val_subj = tr_subj[: max(1, int(0.15 * len(tr_subj)))]
        is_val = np.isin(subj[tr], val_subj)
        va, tr = tr[is_val], tr[~is_val]

        torch.manual_seed(seed + f)
        model = S2DLite(dims, n_classes, T=T, **model_kw).to(dev)
        counts = np.bincount(data["y"][tr], minlength=n_classes).astype(np.float32)
        w = torch.tensor(counts.sum() / (n_classes * np.maximum(counts, 1)), device=dev)
        crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

        mk = lambda idx, train: DataLoader(MecoDataset(data, idx, mods, train), batch_size=bs,
                                           shuffle=train, drop_last=False)
        ltr, lva, lte = mk(tr, True), mk(va, False), mk(te, False)

        best, best_state = -1, None
        for ep in range(epochs):
            model.train()
            for x, miss, y in ltr:
                x, miss, y = _to(x, miss, y, dev)
                loss = crit(model(x, miss), y)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            v = evaluate(model, lva, dev)["UAR"]
            if v > best:
                best, best_state = v, copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        r = evaluate(model, lte, dev)
        print(f"fold {f}: val UAR {best:.3f} | test UAR {r['UAR']:.3f} WAR {r['WAR']:.3f}")
        results.append(r)
    print("mean test UAR %.3f  WAR %.3f" % (np.mean([r["UAR"] for r in results]),
                                           np.mean([r["WAR"] for r in results])))
    return results


if __name__ == "__main__":  # smoke test on random data
    # from meco_data import make_synthetic, normalize_per_subject
    data = normalize_per_subject(make_synthetic())
    train_cv(data, n_classes=6, folds=4, epochs=3, bs=32, au_encoder="conv")

ImportError: cannot import name '_center' from 'numpy.core._multiarray_umath' (/Users/isakhamrin/Programmering/year5/project_course/Scientific-computing-project---Emotion-detection/venv/lib/python3.11/site-packages/numpy/core/_multiarray_umath.cpython-311-darwin.so)

In [38]:
import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import GroupKFold

print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

ImportError: cannot import name '_center' from 'numpy.core._multiarray_umath' (/Users/isakhamrin/Programmering/year5/project_course/Scientific-computing-project---Emotion-detection/venv/lib/python3.11/site-packages/numpy/core/_multiarray_umath.cpython-311-darwin.so)